In [ ]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [ ]:
def search(query: str) -> list[dict]:
    """
    Search the documentation database for relevant results based on a query string.

    Args:
        query (str): The search query to look up in the index.

    Returns:
        list[dict]: A list of search result objects returned by the index.
    """
    results = index.search(
        query=query,
        num_results=5
    )
    return results

def add_entry(filename, title, description, content):
    """
    Add a new documentation entry to the index.

    Args:
        filename (str): The source filename associated with the entry.
        title (str): The title of the documentation entry.
        description (str): A short description summarizing the entry.
        content (str): The full content of the documentation entry.

    Returns:
        str: A status message indicating the result of the operation ("OK").
    """
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

In [ ]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

llm_client=OpenAIClient(
    model='gpt-5.4-nano',
    client=OpenAI()
)

chat_interface=IPythonChatInterface()
runner_callback=DisplayingRunnerCallback(chat_interface=chat_interface)


In [ ]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search

2) in the second interation, analyze the results from the previous search
   and perform 2 more searches

3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [ ]:

from toyaikit import tools

agent_tools=tools.Tools()
agent_tools.add_tool(search)
agent_tools.add_tool(add_entry)
agent_tools.get_tools()
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [ ]:
result = agent.loop(
    'how can I create evidently dahsbard?',
    callback=runner_callback
)

In [ ]:
from searchIndexTools import SearchIndexTools
search_tool_object=SearchIndexTools(index)
results= search_tool_object.search('dashbrd')
results

In [ ]:
agent_tools=tools.Tools()
agent_tools.add_tool(search_tool_object.add_entry)
agent_tools.add_tool(search_tool_object.search)



In [ ]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

result = agent.loop(
    'how can I create evidently dahsbard?',
    callback=runner_callback
)